# Experiment 5: Backward Chaining Engine with Askable Facts

## Aim
To implement a backward chaining engine which is goal-driven reasoning for car-fault diagnosis using askable facts.

## Theory
Backward chaining starts from a **goal** (hypothesis) and works backwards, checking whether the conditions needed to prove it are true. It is **goal-driven**: only facts relevant to the current hypothesis are pursued. This is efficient when the number of possible goals is smaller than the total number of facts.

The engine maintains a **goal stack**. To prove a goal:
1. If the goal is already known (proved or disproved), return immediately.
2. Find a rule whose conclusion matches the goal.
3. Recursively try to prove each condition of that rule.
4. If a condition cannot be derived by any rule, it is **askable** — the user is queried for it.
5. If all conditions are satisfied, the goal is proved.

A **WHY** option lets the user ask which rule is responsible for a question, providing transparency into the reasoning chain.

## Algorithm
1. Define hypotheses (faults) and production rules with conditions.
2. For each hypothesis (in order), attempt to prove it via backward chaining.
3. When an askable fact is needed, prompt the user (yes/no/WHY).
4. On WHY, display the rule that triggered the question and continue.
5. Stop as soon as the first hypothesis is proved; report the diagnosis.

## Knowledge Base (Car-Fault Diagnosis Rules)

| Rule | Conditions | Conclusion (Fault) |
|------|-----------|--------------------|
| R1 | engine does not start, battery is flat | flat battery fault |
| R2 | engine does not start, lights do not work | flat battery fault |
| R3 | engine does not start, engine cranks, fuel is low | fuel fault |
| R4 | engine does not start, engine cranks, fuel is ok, engine floods | carburetor fault |
| R5 | engine does not start, engine cranks, fuel is ok, spark is weak | ignition fault |
| R6 | engine does not start, clicking sound, engine does not crank | starter motor fault |

## Procedure
1. Define the car-fault knowledge base as IF-THEN production rules.
2. Define a list of hypotheses (faults) to test, in priority order.
3. Implement a `prove(goal)` function that:
   - Returns immediately if the goal is already known.
   - Finds a matching rule and tries to prove each condition recursively.
   - Asks the user when a condition is askable (not derivable by any rule).
   - Supports a **WHY** response that displays the responsible rule.
4. Iterate over hypotheses until one is proved.
5. Run the three exercises and observe the results.

## Implementation

In [1]:
# Knowledge Base
# Each rule:
#   'id'         - rule identifier
#   'conditions' - all must be proved for the conclusion to hold
#   'conclusion' - the hypothesis this rule supports

RULES = [
    # Flat battery (battery test)
    {'id': 'R1', 'conditions': ['engine does not start', 'battery is flat'],
     'conclusion': 'flat battery fault'},
    # Flat battery (lights test)
    {'id': 'R2', 'conditions': ['engine does not start', 'lights do not work'],
     'conclusion': 'flat battery fault'},
    # Fuel fault
    {'id': 'R3', 'conditions': ['engine does not start', 'engine cranks', 'fuel is low'],
     'conclusion': 'fuel fault'},
    # Carburetor fault
    {'id': 'R4', 'conditions': ['engine does not start', 'engine cranks',
                                'fuel is ok', 'engine floods'],
     'conclusion': 'carburetor fault'},
    # Ignition fault
    {'id': 'R5', 'conditions': ['engine does not start', 'engine cranks',
                                'fuel is ok', 'spark is weak'],
     'conclusion': 'ignition fault'},
    # Starter motor fault  -- Exercise 1
    {'id': 'R6', 'conditions': ['engine does not start', 'clicking sound',
                                'engine does not crank'],
     'conclusion': 'starter motor fault'},
]

# Hypotheses to test, in priority order
HYPOTHESES = [
    'flat battery fault',
    'fuel fault',
    'carburetor fault',
    'ignition fault',
    'starter motor fault',
]

print('Knowledge base loaded:', len(RULES), 'rules')
print('Hypotheses           :', HYPOTHESES)

Knowledge base loaded: 6 rules
Hypotheses           : ['flat battery fault', 'fuel fault', 'carburetor fault', 'ignition fault', 'starter motor fault']


In [2]:
# Backward Chaining Engine

def backward_chain(hypotheses=None, rules=None, simulate=None):
    """
    Backward chaining engine for car-fault diagnosis.

    Parameters
    ----------
    hypotheses : list of str
        Ordered list of fault hypotheses to test.
    rules : list of dict
        Production rules (id, conditions, conclusion).
    simulate : dict or None
        If provided, maps fact strings to 'yes'/'no' answers
        for non-interactive (automated) runs used in exercises.
        When None, the user is prompted interactively.

    Returns
    -------
    str or None
        The identified fault, or None if no fault is diagnosed.
    """
    if hypotheses is None:
        hypotheses = HYPOTHESES
    if rules is None:
        rules = RULES

    # Working memory: known facts -> True / False
    known = {}

    # Determine which facts are askable (not the conclusion of any rule)
    derived_facts = {r['conclusion'] for r in rules}

    def ask_user(fact, responsible_rule):
        """Ask the user about an askable fact; supports WHY."""
        while True:
            if simulate is not None:
                answer = simulate.get(fact, 'no')
                print(f'  [Q] Is it true that: "{fact}"? --> {answer}')
                return answer.strip().lower() == 'yes'
            else:
                answer = input(
                    f'\n  Is it true that: "{fact}"?\n'
                    f'  (yes / no / WHY): '
                ).strip().lower()

            if answer == 'why':
                # Exercise 3: WHY - display the rule responsible for asking
                print(f'\n  [WHY] I am asking because of rule {responsible_rule["id"]}:')
                print(f'        IF   {" AND ".join(responsible_rule["conditions"])}')
                print(f'        THEN {responsible_rule["conclusion"]}')
                print(f'        I need to confirm "{fact}" to test this hypothesis.\n')
            elif answer in ('yes', 'no'):
                return answer == 'yes'
            else:
                print('  Please enter yes, no, or WHY.')

    def prove(goal, responsible_rule=None):
        """Try to prove goal; return True if proved, False otherwise."""
        # Already known?
        if goal in known:
            return known[goal]

        # Try to prove via rules whose conclusion matches the goal
        matching_rules = [r for r in rules if r['conclusion'] == goal]
        for rule in matching_rules:
            all_conditions_met = True
            for condition in rule['conditions']:
                if not prove(condition, responsible_rule=rule):
                    all_conditions_met = False
                    break
            if all_conditions_met:
                known[goal] = True
                return True

        # No rule can prove it -- askable fact
        if goal not in derived_facts:
            result = ask_user(goal, responsible_rule)
            known[goal] = result
            return result

        # Could not prove via any rule
        known[goal] = False
        return False

    print('=' * 60)
    print('BACKWARD CHAINING ENGINE - CAR FAULT DIAGNOSIS')
    print('=' * 60)
    print('Testing hypotheses in order:', hypotheses)
    print()

    for hypothesis in hypotheses:
        print(f'>> Testing hypothesis: "{hypothesis}"')
        if prove(hypothesis):
            print()
            print('=' * 60)
            print(f'DIAGNOSIS: {hypothesis}')
            print('=' * 60)
            print('Facts established:', {k: v for k, v in known.items() if v})
            return hypothesis
        else:
            print(f'   >> "{hypothesis}" could NOT be confirmed.\n')

    print('=' * 60)
    print('No fault could be diagnosed with the given answers.')
    print('=' * 60)
    return None


print('Engine defined.')

Engine defined.


## Exercise 1 — Starter Motor Fault

Rule **R6** was added:
```
IF   engine does not start
AND  clicking sound
AND  engine does not crank
THEN starter motor fault
```

**Scenario**: The engine does not start, there is a clicking sound, and the engine does not crank.

**Predicted facts asked** (in order, for each hypothesis being tested):
- *flat battery fault* (R1): `engine does not start` → yes, `battery is flat` → **no**
- *flat battery fault* (R2): `lights do not work` → **no** → hypothesis fails
- *fuel fault* (R3): `engine cranks` → **no** → hypothesis fails
- *carburetor fault* (R4): `engine cranks` already known = no → fails
- *ignition fault* (R5): `engine cranks` already known = no → fails
- *starter motor fault* (R6): `clicking sound` → **yes**, `engine does not crank` → **yes** → **PROVED**

Previously answered facts are **reused from working memory** — not asked again.

In [3]:
# Exercise 1 -- Starter motor fault (simulated answers)
starter_motor_answers = {
    'engine does not start': 'yes',
    'battery is flat'      : 'no',
    'lights do not work'   : 'no',
    'engine cranks'        : 'no',
    'fuel is low'          : 'no',
    'fuel is ok'           : 'no',
    'engine floods'        : 'no',
    'spark is weak'        : 'no',
    'clicking sound'       : 'yes',
    'engine does not crank': 'yes',
}

print('Exercise 1 -- Starter Motor Fault Scenario')
print('Simulated answers:', starter_motor_answers)
print()
result = backward_chain(simulate=starter_motor_answers)
print()
print('Observation: The engine established facts about engine not starting and')
print('clicking sound + no crank. Facts from earlier hypotheses (engine does not')
print('start, engine cranks=no) were reused -- not re-asked. The correct fault')
print('"starter motor fault" was identified via rule R6.')

Exercise 1 -- Starter Motor Fault Scenario
Simulated answers: {'engine does not start': 'yes', 'battery is flat': 'no', 'lights do not work': 'no', 'engine cranks': 'no', 'fuel is low': 'no', 'fuel is ok': 'no', 'engine floods': 'no', 'spark is weak': 'no', 'clicking sound': 'yes', 'engine does not crank': 'yes'}

BACKWARD CHAINING ENGINE - CAR FAULT DIAGNOSIS
Testing hypotheses in order: ['flat battery fault', 'fuel fault', 'carburetor fault', 'ignition fault', 'starter motor fault']

>> Testing hypothesis: "flat battery fault"
  [Q] Is it true that: "engine does not start"? --> yes
  [Q] Is it true that: "battery is flat"? --> no
  [Q] Is it true that: "lights do not work"? --> no
   >> "flat battery fault" could NOT be confirmed.

>> Testing hypothesis: "fuel fault"
  [Q] Is it true that: "engine cranks"? --> no
   >> "fuel fault" could NOT be confirmed.

>> Testing hypothesis: "carburetor fault"
   >> "carburetor fault" could NOT be confirmed.

>> Testing hypothesis: "ignition faul

## Exercise 2 — Change Order of Hypotheses

When the **order of hypotheses** changes, the engine tests them in the new order. This affects:
- Which questions are asked first.
- How quickly a diagnosis is reached.

**Scenario**: Same starter-motor facts as Exercise 1, but hypotheses reordered so `starter motor fault` comes **first**.

**Expected observation**: The engine goes straight to testing `starter motor fault` and confirms it immediately — far fewer questions are asked.

In [4]:
# Exercise 2 -- Reordered hypotheses
# Putting 'starter motor fault' first means it is proved with minimal questions.
reordered_hypotheses = [
    'starter motor fault',   # moved to first
    'flat battery fault',
    'fuel fault',
    'carburetor fault',
    'ignition fault',
]

print('Exercise 2 -- Reordered Hypotheses')
print('New hypothesis order:', reordered_hypotheses)
print()
result_reordered = backward_chain(
    hypotheses=reordered_hypotheses,
    simulate=starter_motor_answers
)
print()
print('Observation: With "starter motor fault" listed first, only the three facts')
print('needed for R6 are asked (engine does not start, clicking sound, engine does')
print('not crank). The engine terminates earlier -- demonstrating how hypothesis')
print('ordering directly controls which questions are posed and how quickly a')
print('diagnosis is reached.')

Exercise 2 -- Reordered Hypotheses
New hypothesis order: ['starter motor fault', 'flat battery fault', 'fuel fault', 'carburetor fault', 'ignition fault']

BACKWARD CHAINING ENGINE - CAR FAULT DIAGNOSIS
Testing hypotheses in order: ['starter motor fault', 'flat battery fault', 'fuel fault', 'carburetor fault', 'ignition fault']

>> Testing hypothesis: "starter motor fault"
  [Q] Is it true that: "engine does not start"? --> yes
  [Q] Is it true that: "clicking sound"? --> yes
  [Q] Is it true that: "engine does not crank"? --> yes

DIAGNOSIS: starter motor fault
Facts established: {'engine does not start': True, 'clicking sound': True, 'engine does not crank': True, 'starter motor fault': True}

Observation: With "starter motor fault" listed first, only the three facts
needed for R6 are asked (engine does not start, clicking sound, engine does
not crank). The engine terminates earlier -- demonstrating how hypothesis
ordering directly controls which questions are posed and how quickly a

## Exercise 3 — WHY Option

When the user types **WHY** at a question prompt, the engine displays:
- The rule ID responsible for asking the question.
- The full IF-THEN rule text.
- Which fact it is trying to confirm.

This is implemented in `ask_user()` — the `responsible_rule` parameter carries the rule that is currently being evaluated when the askable fact is needed.

**Interactive demo**: Run the cell below. At any `yes / no / WHY` prompt, type `WHY` to see the explanation, then answer `yes` or `no` to continue.

In [5]:
# Exercise 3 -- WHY option (interactive)
# Run this cell to interact with the engine.
# Type WHY at any prompt to see which rule is asking the question.

print('Exercise 3 -- Interactive WHY Demo')
print('At any prompt type WHY to see the rule responsible for the question.')
print('Then answer yes or no to continue.\n')
backward_chain()  # No simulate= -> interactive mode

Exercise 3 -- Interactive WHY Demo
At any prompt type WHY to see the rule responsible for the question.
Then answer yes or no to continue.

BACKWARD CHAINING ENGINE - CAR FAULT DIAGNOSIS
Testing hypotheses in order: ['flat battery fault', 'fuel fault', 'carburetor fault', 'ignition fault', 'starter motor fault']

>> Testing hypothesis: "flat battery fault"

DIAGNOSIS: flat battery fault
Facts established: {'engine does not start': True, 'battery is flat': True, 'flat battery fault': True}


'flat battery fault'

## Exercise 3 — WHY Option (Simulated Trace)

The cell below demonstrates what the engine prints when the user selects **WHY** for two different questions.

In [6]:
# Simulated WHY output -- demonstrates what the WHY response looks like

def demonstrate_why():
    """Print a representative WHY explanation for illustrative purposes."""
    # WHY for rule R1 asking about 'battery is flat'
    rule_r1 = RULES[0]
    fact_asked = 'battery is flat'
    print('  [Simulated user types WHY at the prompt for: "battery is flat"]')
    print()
    print(f'  [WHY] I am asking because of rule {rule_r1["id"]}:')
    print(f'        IF   {" AND ".join(rule_r1["conditions"])}')
    print(f'        THEN {rule_r1["conclusion"]}')
    print(f'        I need to confirm "{fact_asked}" to test this hypothesis.')
    print()

    # WHY for rule R6 asking about 'clicking sound'
    rule_r6 = RULES[5]
    fact_asked2 = 'clicking sound'
    print('  [Simulated user types WHY at the prompt for: "clicking sound"]')
    print()
    print(f'  [WHY] I am asking because of rule {rule_r6["id"]}:')
    print(f'        IF   {" AND ".join(rule_r6["conditions"])}')
    print(f'        THEN {rule_r6["conclusion"]}')
    print(f'        I need to confirm "{fact_asked2}" to test this hypothesis.')


print('Exercise 3 -- WHY Option Demonstration')
print('=' * 60)
demonstrate_why()
print()
print('Observation: The WHY option displays the rule ID, its full IF-THEN')
print('form, and the specific fact being queried. This gives the user')
print("transparency into the engine's goal-directed reasoning.")

Exercise 3 -- WHY Option Demonstration
  [Simulated user types WHY at the prompt for: "battery is flat"]

  [WHY] I am asking because of rule R1:
        IF   engine does not start AND battery is flat
        THEN flat battery fault
        I need to confirm "battery is flat" to test this hypothesis.

  [Simulated user types WHY at the prompt for: "clicking sound"]

  [WHY] I am asking because of rule R6:
        IF   engine does not start AND clicking sound AND engine does not crank
        THEN starter motor fault
        I need to confirm "clicking sound" to test this hypothesis.

Observation: The WHY option displays the rule ID, its full IF-THEN
form, and the specific fact being queried. This gives the user
transparency into the engine's goal-directed reasoning.


## Conclusion

The backward chaining engine successfully implements goal-driven reasoning for car-fault diagnosis using askable facts.

- **Exercise 1**: Adding rule **R6** (starter motor fault — clicking sound + engine not cranking) allowed the engine to correctly diagnose a starter motor fault. Facts already established (e.g. `engine does not start`) were reused from working memory and not re-asked.

- **Exercise 2**: Changing the order of hypotheses directly changes which questions are asked and when. Placing `starter motor fault` first caused the engine to confirm the fault after only three questions, instead of exhausting all other hypotheses first.

- **Exercise 3**: The **WHY** option provides a transparent explanation by displaying the rule ID, its IF-THEN structure, and the specific fact being pursued. This improves user trust and understanding of the reasoning process.